# Vibronic coupling in tetracene dimers
## General post-processing / prototyping notebook
### Can be done locally:
1. Boys localisation and partial diagonalisation (requires calc.chk)
    - reconstruct Fock matrix (takes some time)
    - localize frontier orbitals
    - visualize frontier orbitals (careful with notebook size)

2. Electron-phonon coupling analysis (requires out.h5)
    - transform e-ph coupling to frontier orbital space
    - visualize molecular distortions
    - calculate effective exchange vibronic coupling in the TT subspace from two-electron integrals and one-electron LVC [Kollmar, J. Chem. Phys. 98 (1993)]
    - reduce dimensionality of the e-ph coupling (clustering, SVD)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
import py3Dmol
from pyscf import gto, dft
from pyscfutils import xyz_string, mf_from_chk, Boys_frontier_orbitals, \
                       pair_neighboring_orbitals, block_frontier_orbitals, \
                       fock_matrix, two_electron_integrals, cartesian_modevec, \
                       multiconfig_vibronic_coupling, save_orbital
from visualutils import view_orbital, view_vibration

# Constants
HARTREE2EV = 27.2114079527
HARTREE2ICM = 219474.63

# Directories for source data and results
DATADIR = os.path.realpath("../runs/10264877/")
SAVEDIR = os.path.realpath("../results/")


## 1. Import and post-process DFT calculation

Import ```mf``` and ```mol``` object from checkfile.

In [2]:
mf = mf_from_chk(DATADIR + "/" + "calc.chk")
mol = mf.mol

Calculate Fock matrix in the atomic orbital basis (time consuming, takes about 2 minutes for a tetracene dimer).

In [3]:
fock_ao = fock_matrix(mf)

Localize the 4 frontier molecular orbitals using Boys localisation. We expect two Boys orbitals on each tetracene fragment. Reorder the molecular orbitals so that they appear one pair after another. After that, block-diagonalize the Fock matrix to obtain localized orthogonal orbitals corresponding to HOMO and LUMO of each tetracene unit.

In [4]:
mo_coeff_boys, idx = Boys_frontier_orbitals(mf, 4)
mo_coeff_boys, idx = pair_neighboring_orbitals(mf, mo_coeff_boys)
mo_coeff_block = block_frontier_orbitals(mf, mo_coeff_boys, fock=fock_ao)

Fock matrix in the frontier subspace in different bases: MO, Boys, block-diagonal

In [5]:
fock_mo = mf.mo_coeff[:,idx].T @ fock_ao @ mf.mo_coeff[:,idx]
fock_boys = mo_coeff_boys.T @ fock_ao @ mo_coeff_boys
fock_block = mo_coeff_block.T @ fock_ao @ mo_coeff_block
print(fock_mo * HARTREE2EV * 1000, '\n')
print(fock_boys * HARTREE2EV * 1000, '\n')
print(fock_block * HARTREE2EV * 1000)

[[-2.66816666e+05  8.94002053e-04 -2.30493895e-02  8.59842612e-04]
 [ 8.94002053e-04 -2.66467987e+05  2.29200105e-02  6.11908028e-03]
 [-2.30493894e-02  2.29200105e-02 -2.66817304e+05  2.41600915e-02]
 [ 8.59842612e-04  6.11908030e-03  2.41600915e-02 -2.66467920e+05]] 

[[-3786.46677253   876.78171228   -84.31604725   -30.61643776]
 [  876.78171228 -3786.44798253   -30.61867477   -84.3157326 ]
 [  -84.31604725   -30.61867477 -3786.37914255   876.86612842]
 [  -30.61643776   -84.3157326    876.86612842 -3786.37804852]] 

[[-4.66323909e+03  9.35007155e-13 -5.36983337e+01  3.62150011e-04]
 [-6.42608273e-13 -2.90967567e+03 -1.02398262e-03 -1.14933446e+02]
 [-5.36983337e+01 -1.02398262e-03 -4.66324472e+03 -7.65897801e-13]
 [ 3.62150011e-04 -1.14933446e+02 -2.36389474e-12 -2.90951247e+03]]


### Save and visualize molecular orbitals
Save orbitals as ```.cube``` and ```.html``` files. These can be rendered by py3Dmol inside a Jupyter notebook, or opened in an external browser. Visualization in an external browser helps keep the notebook nice and lightweight (strongly recommended if tracked by git).

In [6]:
# Grid for cube file: low-res (100,50,50), hi-res (400,100,100) 
grid = (100, 50, 50)

# Boys orbitals
for i, mo in enumerate(mo_coeff_boys.T):
    orbname = SAVEDIR + "/" + "orbBoys" + str(i+1) + ".cube"
    save_orbital(mol, orbname, mo, grid=grid)
    v = view_orbital(mol, orbname, iso=0.02, alpha=0.9, html=True)

# Block-diagonalized orbitals
for i, mo in enumerate(mo_coeff_block.T):
    orbname = SAVEDIR + "/" + "orbBlock" + str(i+1) + ".cube"
    save_orbital(mol, orbname, mo, grid=grid)
    v = view_orbital(mol, orbname, iso=0.02, alpha=0.9, html=True)

### Calculate two-electron integrals
Calculate two-electron integrals in the frontier subspace ```integral2e(i,j,k,l)```, corresponding to
$( i j | k l ) = \langle i k | j l \rangle $.
This is quite time consuming (takes approximately 5 minutes for a tetracene dimer).

In [8]:
integral2e = two_electron_integrals(mol, mo_coeff_block)

In [11]:
J_hAlA = integral2e[0,0,1,1] * HARTREE2EV * 1000
J_hBlB = integral2e[2,2,3,3] * HARTREE2EV * 1000
K_hAlA = integral2e[0,1,1,0] * HARTREE2EV * 1000
K_hBlB = integral2e[2,3,3,2] * HARTREE2EV * 1000
print(J_hAlA, J_hBlB)
print(K_hAlA, K_hBlB)

4924.982199111849 4924.963328753594
969.1864487948352 969.2084536462701


According to Smith & Michl, Chem. Rev. 110, 6891-6936 (2010), the CI Hamiltonian matrix elements within the subspace spanned by singlet excitons (SE), charge transfer states (CT), and singlet triplet-triplet (TT) are
$$
\begin{align}
\langle \mathrm{CT}_A|H|\mathrm{CT}_B\rangle &= 2(h_Al_B|l_Ah_B)-(h_Ah_B|l_Al_B) \\
\langle \mathrm{SE}_A|H|\mathrm{SE}_B\rangle &= 2(h_Al_A|l_Bh_B)-(h_Ah_B|l_Bl_A) \\
\langle \mathrm{CT}_A|H|\mathrm{SE}_A\rangle &= \langle l_A|F|l_B \rangle+2(h_Al_B|l_Ah_A)-(h_Ah_A|l_Al_B) \\
\langle \mathrm{CT}_A|H|\mathrm{SE}_B\rangle &= \langle h_A|F|h_B \rangle+2(h_Bl_B|l_Bh_A)-(h_Bh_A|l_Bl_B) \\
\langle \mathrm{SE}_A|H|\mathrm{TT}\rangle &= \sqrt{\frac{3}{2}} \left[(l_Ah_B|l_Bl_A)-(h_Al_B|h_Bh_A)\right]\\
\langle \mathrm{CT}_A|H|\mathrm{TT}\rangle &= \sqrt{\frac{3}{2}} \left[ \langle l_A|F|h_B\rangle + (l_Ah_B|l_Bl_B)-(l_Ah_B|h_Ah_A) \right]
\end{align}
$$
(all other matrix elements can be obtained by switching $A\leftrightarrow B$) and using hermiticity)

## 2. Import and post-process e-ph coupling calculation
Convert e-ph couplings to MO subspace, and then convert that to multiconfigurational singlet subspace spanned by symmetric/antisymmetric singlet exciton ($\mathrm{SE}_\pm$), charge transfer ($\mathrm{CT}_\pm$) and correlated triplet ($\mathrm{TT}$) states.

In [7]:
# Import e-ph data
h5file = DATADIR + "/" + "dft-lvc-out.h5"
with h5py.File(h5file, "r") as f:
    ephmat  = np.array(list(f["ephmat"])) # hartree
    omega   = np.array(list(f["omega"])) # cm^-1
    modevec = np.array(list(f["modevec"])) # mass-weighted

# Convert ephmat from AO to MO subspace (units cm^-1)
g = np.einsum('kij, ip, jq -> kpq', ephmat,
                                    mo_coeff_block, 
                                    mo_coeff_block, 
                                    optimize=True) * HARTREE2ICM

# Convert from MO subspace to multiconfigurational singlet subspace
W = multiconfig_vibronic_coupling(g)

In [46]:
view_vibration(mol, modevec[:,2], amplitude=.5, equilibrium=True).show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# TO DO:
## Continue transferring code from visualize.ipynb to util modules

## Idea:
### We already have the vibronic coupling in the full MO basis - let's use it!

Instead of doing SVD to identify the vibronic coupling patterns in a small pre-selected electroinc state manifold, we could use the information from more than those four frontier molecular orbitals?

This is the same as identifying a preferred vibronic basis?
Identify hieararchies with SVD.